# "Računalniški vid · Od začetka"

**Stopnja** 1 · **Predviden Čas** okoli 20-30 minut

> Bralec bo spoznal, kako moramo sliko prikazati računalniku, da jo ta razume. Ter glavne principe za manipulacijo in obdelavo slik. Hkrati bo ta uvod služil kot motivacija in tranzicija v svet umetne inteligence v računalniškem vidu.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aicademy-si/notebooks/blob/dev/modules/03-computer-vision/2-hook/hook.ipynb)

## Začnimo na koncu

Zakaj bi nas sploh računalniški vid zanimal? Kaj z njim premoremo danes?

Na osnovi računalniškega vida danes:
- **roboti** izvajajo akrobacije in se nemoteno gibljejo čez zapletene ovire;
- **avtomobili** postajajo povsem samovozeči;
- **medicinska stroka** odkriva bolezni pri pacientih.

Za video pregled današnjih uporab računalniškega vida smo ti pustili linke do video povezav.

<table><tr>
<td align="center" width="33%"><a href="https://www.youtube.com/watch?v=tF4DML7FIWk"><img src="https://img.youtube.com/vi/tF4DML7FIWk/maxresdefault.jpg" width="100%"></a><br><b>Boston Dynamics</b></td>
<td align="center" width="33%"><a href="https://www.youtube.com/watch?v=tlThdr3O5Qo"><img src="https://img.youtube.com/vi/tlThdr3O5Qo/maxresdefault.jpg" width="100%"></a><br><b>Tesla</b></td>
<td align="center" width="33%"><a href="https://www.youtube.com/watch?v=MCI0xEGvHx8"><img src="https://img.youtube.com/vi/MCI0xEGvHx8/maxresdefault.jpg" width="100%"></a><br><b>Google DeepMind</b></td>
</tr></table>

- ▶ Boston Dynamics: [Atlas | Partners in Parkour](https://www.youtube.com/watch?v=tF4DML7FIWk)
- ▶ Tesla: [Full Self-Driving](https://www.youtube.com/watch?v=tlThdr3O5Qo)
- ▶ Google DeepMind: [DeepMind Health Research and Moorfields Eye Hospital: What our research shows](https://www.youtube.com/watch?v=MCI0xEGvHx8)

In [69]:
#@title Poženi celico (klikni ▶ na levi, ali Shift+Enter). Naložile se bodo slike in orodja. Kode ni potrebno brati.
import base64
import io
import json
import uuid

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image, ImageDraw, ImageFont, ImageOps
from scipy.ndimage import correlate
from skimage import color, data, transform

# Every word a student sees in a picture or a message, in one place, so that it is easy to translate.
LABELS = {
    "cat": "mačka",
    "astronaut": "astronavt",
    "rocket": "raketa",
    "coffee": "kava",
    "original": "slika",
    "filtered": "po filtru",
    "red": "rdeča",
    "green": "zelena",
    "blue": "modra",
    "squares across": "kvadratkov v vrsti",
    "nothing": "brez spremembe",
    "blur": "zameglitev",
    "sharpen": "izostritev",
    "vertical edges": "navpični robovi",
    "horizontal edges": "vodoravni robovi",
    "all edges": "vsi robovi",
    "handmade": "naš filter",
    "handmade result": "po našem filtru",
    "learned": "najbolj podoben filter mreže",
    "learned result": "po filtru mreže",
    "only in colab": "Nalaganje slik deluje v Google Colabu. Tukaj uporabi eno od vgrajenih slik.",
    "secret": "AI",
    "horizontal": "vodoravne črte",
    "vertical": "navpične črte",
    "correct": "pravilno",
    "bravo": "Bravo! Model loči vse slike.",
    "threshold": "prag",
    "average": "povprečna svetlost po filtru",
}

MAX_SIDE = 320  # bigger photos are shrunk to this many pixels on their longer side, so everything stays fast


def fit(rgb):
    """Shrink a picture so that its longer side has at most MAX_SIDE pixels."""
    h, w = rgb.shape[:2]
    scale = MAX_SIDE / max(h, w)
    if scale < 1:
        rgb = transform.resize(rgb, (round(h * scale), round(w * scale)), anti_aliasing=True, preserve_range=True)
    return np.round(rgb).astype(np.uint8)


PICTURES = {
    "cat": fit(data.chelsea()),
    "astronaut": fit(data.astronaut()),
    "rocket": fit(data.rocket()),
    "coffee": fit(data.coffee()),
}


def choose_picture(name):
    """A built-in picture, by its English name or by its name in LABELS ("cat" or "mačka")."""
    names = {LABELS[key]: key for key in PICTURES}
    return PICTURES[names.get(name, name)]


def upload_picture():
    """Ask for a photo from your computer or phone. Works in Google Colab."""
    try:
        from google.colab import files
    except ImportError:
        print(LABELS["only in colab"])
        return None
    uploaded = files.upload()
    if not uploaded:
        return None
    photo = Image.open(io.BytesIO(next(iter(uploaded.values()))))
    photo = ImageOps.exif_transpose(photo).convert("RGB")  # phone photos stand the right way up
    return fit(np.array(photo))


def to_grey(rgb):
    """One number per pixel: 0 is black, 255 is white."""
    return np.round(color.rgb2gray(rgb) * 255).astype(np.uint8)


def show(rgb):
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb)
    plt.axis("off")
    plt.show()


def _number(v):
    return f"{v:.2f}".rstrip("0").rstrip(".") if isinstance(v, float) else str(v)


def _cells(ax, values, shades=None, fontsize=9):
    """A grid with one number in each cell. `shades` (0 to 255) colours the cells grey; without it they are white."""
    values = np.asarray(values)
    if shades is None:
        shades = np.full(values.shape, 255)
    ax.imshow(shades, cmap="gray", vmin=0, vmax=255)
    for (i, j), v in np.ndenumerate(values):
        ax.text(j, i, _number(v.item()), ha="center", va="center", fontsize=fontsize,
                color="white" if shades[i, j] < 128 else "black")
    ax.set_xticks(np.arange(-0.5, values.shape[1]), minor=True)
    ax.set_yticks(np.arange(-0.5, values.shape[0]), minor=True)
    ax.grid(which="minor", color="0.6", linewidth=0.5)
    ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)


def _png(fig):
    """The figure as PNG bytes, for an image widget. Drawing into an image widget works the same in Colab, VS Code and Jupyter."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=80, bbox_inches="tight")
    plt.close(fig)
    return buffer.getvalue()


def _picture_widget(width="100%"):
    return widgets.Image(format="png", layout=widgets.Layout(width=width, max_width="100%", height="auto"))


def _live(draw, controls, width="100%"):
    """Show the controls, and below them the figure from `draw`, drawn again whenever a control changes."""
    picture = _picture_widget(width)

    def update(change=None):
        picture.value = _png(draw(**{name: control.value for name, control in controls.items()}))

    for control in controls.values():
        control.observe(update, names="value")
    update()
    display(widgets.VBox([*controls.values(), picture]))


def _busiest(grey, size):
    """The top-left corner of the size×size square whose pixels differ the most: usually something worth a look."""
    spread = np.lib.stride_tricks.sliding_window_view(grey.astype(float), (size, size)).std(axis=(2, 3))
    return np.unravel_index(np.argmax(spread), spread.shape)


# The page as_numbers shows. It runs in the browser, so pointing at a number answers at once, without Python.
_NUMBERS_HTML = """
<div id="UID" class="numbers-view">
  <style>
    #UID { background: #fff; color: #000; padding: 12px; font-family: sans-serif; border-radius: 6px; }
    #UID .controls { margin-bottom: 10px; font-size: 14px; }
    #UID .controls input { vertical-align: middle; width: 220px; }
    #UID .panels { display: flex; gap: 16px; flex-wrap: wrap; align-items: flex-start; }
    #UID .numbers, #UID .squares { display: grid; cursor: crosshair; user-select: none; }
    #UID .numbers div { display: flex; align-items: center; justify-content: center; color: #444; }
    #UID .numbers div.on { outline: 2px solid red; outline-offset: -1px; color: #000; font-weight: bold; }
    #UID .squares div.on { outline: 2px solid red; outline-offset: -1px; }
    #UID .side { display: flex; flex-direction: column; gap: 12px; }
    #UID .whole { position: relative; width: 160px; line-height: 0; }
    #UID .whole img { width: 100%; }
    #UID .mark { position: absolute; outline: 2px solid red; }
    #UID .readout { width: 80px; height: 80px; display: flex; align-items: center; justify-content: center;
                    font-size: 28px; font-weight: bold; border: 1px solid #999; }
  </style>
  <div class="controls">LABEL: <input type="range" min="0" max="LAST" step="1" value="START_INDEX"> <b class="size"></b></div>
  <div class="panels">
    <div class="numbers"></div>
    <div class="squares"></div>
    <div class="side">
      <div class="whole"><img src="data:image/png;base64,THUMBNAIL"><div class="mark"></div></div>
      <div class="readout"></div>
    </div>
  </div>
  <script>
  (() => {
    const root = document.getElementById("UID");
    const grids = GRIDS;  // [squares across, rows of numbers] for every slider position
    const slider = root.querySelector("input"), sizeText = root.querySelector(".size");
    const numbers = root.querySelector(".numbers"), squares = root.querySelector(".squares");
    const mark = root.querySelector(".mark"), readout = root.querySelector(".readout");
    let grid = null, at = START_PIXEL;

    function build() {
      const [n, g] = grids[+slider.value];
      if (grid) {  // keep pointing at the same place in the picture
        at = [Math.floor((at[0] + 0.5) / grid.length * g.length), Math.floor((at[1] + 0.5) / grid[0].length * g[0].length)];
      }
      grid = g;
      sizeText.textContent = n;
      const rows = g.length, cols = g[0].length, cell = Math.floor(PANEL / Math.max(rows, cols));
      for (const box of [numbers, squares]) {
        box.style.gridTemplateColumns = `repeat(${cols}, ${cell}px)`;
        box.style.gridAutoRows = `${cell}px`;
        box.replaceChildren();
        g.forEach((row, r) => row.forEach((v, c) => {
          const d = document.createElement("div");
          d.dataset.r = r;
          d.dataset.c = c;
          if (box === numbers) {
            d.textContent = v;
            d.style.fontSize = `${cell * 0.42}px`;
          } else {
            d.style.background = `rgb(${v},${v},${v})`;
          }
          box.appendChild(d);
        }));
      }
      show(...at);
    }

    function show(r, c) {
      at = [r, c];
      root.querySelectorAll(".on").forEach(d => d.classList.remove("on"));
      const rows = grid.length, cols = grid[0].length, i = r * cols + c;
      numbers.children[i].classList.add("on");
      squares.children[i].classList.add("on");
      Object.assign(mark.style, {left: `${c / cols * 100}%`, top: `${r / rows * 100}%`,
                                 width: `${100 / cols}%`, height: `${100 / rows}%`});
      const v = grid[r][c];
      readout.textContent = v;
      readout.style.background = `rgb(${v},${v},${v})`;
      readout.style.color = v < 128 ? "#fff" : "#000";
    }

    for (const box of [numbers, squares]) {
      for (const event of ["mouseover", "click"]) {
        box.addEventListener(event, e => {
          const d = e.target.closest("[data-r]");
          if (d) show(+d.dataset.r, +d.dataset.c);
        });
      }
    }
    slider.addEventListener("input", build);
    build();
  })();
  </script>
</div>
"""

NUMBER_SIZES = list(range(8, 65, 4))  # the slider's steps: how many squares across


def as_numbers(rgb, size=32):
    """Each pixel as its brightness number, next to the same pixels as greys and the whole picture.
    Point at a number or a square to see the same pixel everywhere; the slider sets how many squares across."""
    grey = to_grey(rgb)
    h, w = grey.shape
    grids = []
    for n in NUMBER_SIZES:
        scale = n / max(h, w)
        small = transform.resize(grey, (max(3, round(h * scale)), max(3, round(w * scale))),
                                 anti_aliasing=True, preserve_range=True)
        grids.append([n, np.round(small).astype(int).tolist()])
    start = min(range(len(NUMBER_SIZES)), key=lambda i: abs(NUMBER_SIZES[i] - size))
    row, col = np.array(_busiest(np.array(grids[start][1]), 3)) + 1  # start in the middle of the busiest 3×3 spot

    thumbnail = io.BytesIO()
    Image.fromarray(rgb).save(thumbnail, format="png")
    page = _NUMBERS_HTML
    for key, value in {
        "UID": f"numbers-{uuid.uuid4().hex}",
        "LAST": str(len(NUMBER_SIZES) - 1),
        "START_INDEX": str(start),
        "START_PIXEL": json.dumps([int(row), int(col)]),
        "GRIDS": json.dumps(grids),
        "PANEL": "460",  # pixels across each of the two grids
        "LABEL": LABELS["squares across"],
        "THUMBNAIL": base64.b64encode(thumbnail.getvalue()).decode(),
    }.items():
        page = page.replace(key, value)
    display(HTML(page))


def colour_mixer():
    """Three numbers make one colour."""
    def draw(red, green, blue):
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.imshow(np.array([[[red, green, blue]]], dtype=np.uint8))
        ax.set_title(f"({red}, {green}, {blue})")
        ax.axis("off")
        return fig

    _live(draw, {name: widgets.IntSlider(value=0, min=0, max=255, description=name[0].upper(), continuous_update=False)
                 for name in ("red", "green", "blue")}, width="240px")


def channels(rgb):
    """A colour picture is three grey pictures: how much red, green and blue each pixel has."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(rgb)
    axes[0].set_title(LABELS["original"])
    for k, (ax, name, pure) in enumerate(zip(axes[1:], ("red", "green", "blue"), ((1, 0, 0), (0, 1, 0), (0, 0, 1)))):
        ax.imshow(rgb[..., k], cmap=LinearSegmentedColormap.from_list(name, ["black", pure]), vmin=0, vmax=255)
        ax.set_title(LABELS[name])
    for ax in axes:
        ax.axis("off")
    plt.show()


# The page colour_magnifier shows. It runs in the browser, so the square follows the cursor at once, without Python.
_COLOUR_HTML = """
<div id="UID" class="colour-view">
  <style>
    #UID { background: #fff; color: #000; padding: 12px; font-family: sans-serif; border-radius: 6px; }
    #UID .panels { display: flex; gap: 16px; flex-wrap: wrap; align-items: flex-start; }
    #UID .picture { position: relative; line-height: 0; }
    #UID canvas { image-rendering: pixelated; cursor: crosshair; touch-action: none; }
    #UID .mark { position: absolute; outline: 2px solid red; pointer-events: none; }
    #UID .cells { display: grid; grid-template-columns: repeat(SIZE, CELLpx); grid-auto-rows: CELLpx; }
    #UID .cells div { display: flex; align-items: center; justify-content: center; text-align: left;
                      white-space: pre; font-size: FONTpx; line-height: 1.15; font-family: monospace; }
  </style>
  <div class="panels">
    <div class="picture"><canvas></canvas><div class="mark"></div></div>
    <div class="cells"></div>s
  </div>
  <script>
  (() => {
    const root = document.getElementById("UID");
    const size = SIZE, canvas = root.querySelector("canvas");
    const mark = root.querySelector(".mark"), cells = root.querySelector(".cells");
    const img = new Image();
    img.onload = () => {
      const w = img.width, h = img.height;
      canvas.width = w;
      canvas.height = h;
      canvas.style.width = `${PANEL * w / Math.max(w, h)}px`;
      const context = canvas.getContext("2d");
      context.drawImage(img, 0, 0);
      const pixels = context.getImageData(0, 0, w, h).data;
      for (let k = 0; k < size * size; k++) cells.appendChild(document.createElement("div"));

      function show(row, col) {
        row = Math.max(0, Math.min(h - size, row));
        col = Math.max(0, Math.min(w - size, col));
        Object.assign(mark.style, {left: `${col / w * 100}%`, top: `${row / h * 100}%`,
                                   width: `${size / w * 100}%`, height: `${size / h * 100}%`});
        for (let i = 0; i < size; i++) {
          for (let j = 0; j < size; j++) {
            const p = ((row + i) * w + col + j) * 4, r = pixels[p], g = pixels[p + 1], b = pixels[p + 2];
            const d = cells.children[i * size + j];
            d.textContent = `R ${r}\nG ${g}\nB ${b}`;
            d.style.background = `rgb(${r},${g},${b})`;
            d.style.color = 0.299 * r + 0.587 * g + 0.114 * b < 128 ? "#fff" : "#000";
          }
        }
      }

      // The square follows the cursor while it is on the picture, and stays where it was when the cursor leaves.
      canvas.addEventListener("pointermove", e => {
        const box = canvas.getBoundingClientRect();
        const x = (e.clientX - box.left) / box.width * w, y = (e.clientY - box.top) / box.height * h;
        show(Math.round(y - size / 2), Math.round(x - size / 2));
      });
      show(START_ROW, START_COL);
    };
    img.src = "data:image/png;base64,PICTURE";
  })();
  </script>
</div>
"""


def colour_magnifier(rgb, size=10):
    """The picture, and the pixels inside the red square with their three numbers each.
    The square follows the cursor over the picture."""
    row, col = _busiest(to_grey(rgb), size)  # start where there is something to see
    picture = io.BytesIO()
    Image.fromarray(rgb).save(picture, format="png")
    cell = 48  # pixels across each enlarged pixel
    page = _COLOUR_HTML
    for key, value in {
        "UID": f"colour-{uuid.uuid4().hex}",
        "SIZE": str(size),
        "CELL": str(cell),
        "FONT": str(round(cell * 0.24)),
        "PANEL": "460",  # pixels across the picture
        "START_ROW": str(int(row)),
        "START_COL": str(int(col)),
        "PICTURE": base64.b64encode(picture.getvalue()).decode(),
    }.items():
        page = page.replace(key, value)
    display(HTML(page))


def apply_filter(grey, kernel):
    """Each new pixel is the sum of its 3×3 neighbourhood multiplied, place by place, by the filter.
    Results below 0 become 0 and above 255 become 255, so that the result is a picture again."""
    result = correlate(grey.astype(float), np.asarray(kernel, dtype=float), mode="nearest")
    return np.round(np.clip(result, 0, 255)).astype(np.uint8)


NOTHING = [[0, 0, 0], [0, 1, 0], [0, 0, 0]]


class FilterEditor:
    """Nine boxes for the filter's numbers. The picture is filtered again whenever a number changes."""

    def __init__(self, rgb, start=NOTHING):
        self.grey = to_grey(rgb)
        self.boxes = [widgets.FloatText(value=v, step=1, continuous_update=True, layout=widgets.Layout(width="70px"))
                      for v in np.ravel(start)]
        self.picture = _picture_widget()
        for widget in self.boxes:
            widget.observe(lambda change: self.draw(), names="value")
        grid = widgets.GridBox(self.boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 75px)"))
        self.draw()  # before showing it, so that the picture is there from the start
        display(widgets.VBox([grid, self.picture]))

    def kernel(self):
        return np.array([box.value for box in self.boxes]).reshape(3, 3)

    def draw(self):
        fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5))
        left.imshow(self.grey, cmap="gray", vmin=0, vmax=255)
        left.set_title(LABELS["original"])
        right.imshow(apply_filter(self.grey, self.kernel()), cmap="gray", vmin=0, vmax=255)
        right.set_title(LABELS["filtered"])
        for ax in (left, right):
            ax.axis("off")
        self.picture.value = _png(fig)


def filter_editor(rgb, start=NOTHING):
    return FilterEditor(rgb, start)


def secret_picture(message=None, height=140, width=420):
    """A grey picture with a hidden message. The background brightens slowly from left to right; inside the letters,
    a very fine checkerboard makes neighbouring pixels differ by 20 levels, too fine for the eye to see."""
    font_path = font_manager.findfont(font_manager.FontProperties(family="DejaVu Sans", weight="bold"))
    mask = Image.new("L", (width, height), 0)
    ImageDraw.Draw(mask).text((width / 2, height / 2), message or LABELS["secret"], fill=255,
                              font=ImageFont.truetype(font_path, int(height * 0.8)), anchor="mm")
    letters = np.asarray(mask) > 128
    rows, cols = np.indices((height, width))
    background = np.round(50 + 150 * (cols + 0.5 * rows) / (width + 0.5 * height))
    checkerboard = np.where((rows + cols) % 2 == 0, 10, -10)
    grey = np.clip(background + checkerboard * letters, 0, 255).astype(np.uint8)
    return np.stack([grey] * 3, axis=-1)


LINE_KINDS = {1: ("horizontal", "vertical")}  # the two kinds of pictures at each level


def _lines(kind, rng, size=24):
    """A small grey picture with 2 to 4 lines of one kind, on a background of random brightness, with a little noise."""
    background = rng.uniform(40, 215)
    picture = np.full((size, size), background)
    rows, cols = np.indices((size, size))
    along = {"horizontal": rows, "vertical": cols}[kind]
    lowest, highest = along.min(), along.max()
    for _ in range(rng.integers(2, 5)):
        start, thickness = rng.integers(lowest, highest - 1), rng.integers(1, 3)
        contrast = rng.uniform(60, 100) * rng.choice([-1, 1])  # lighter or darker than the background
        picture[(along >= start) & (along < start + thickness)] = background + contrast
    picture += rng.normal(0, 8, picture.shape)
    return np.clip(np.round(picture), 0, 255).astype(np.uint8)


def line_pictures(level=1, count=100):
    """`count` small pictures, half of each kind, always the same ones, and which kind each is (0 or 1)."""
    rng = np.random.default_rng(level)
    kinds = np.arange(count) % 2
    return [_lines(LINE_KINDS[level][k], rng) for k in kinds], kinds


def _best_threshold(averages, kinds):
    """The rule "above the threshold is one kind, below it the other" that gets the most pictures right.
    The computer tries every threshold, and both ways round: this is the model's only learning."""
    order = np.sort(averages)
    candidates = np.concatenate([[order[0] - 1], (order[:-1] + order[1:]) / 2, [order[-1] + 1]])
    best = (0, candidates[0], 1)
    for threshold in candidates:
        above = averages > threshold
        for high_kind in (0, 1):
            correct = int(np.sum(above == (kinds == high_kind)))
            if correct > best[0]:
                best = (correct, threshold, high_kind)
    return best


class ClassifierChallenge:
    """A model that tells two kinds of line pictures apart: your filter, then the average brightness, then a threshold.
    The nine boxes are the filter; the score is drawn again whenever a number changes."""

    def __init__(self, level=1, start=NOTHING):
        self.level = level
        self.pictures, self.kinds = line_pictures(level)
        self.boxes = [widgets.FloatText(value=v, step=1, continuous_update=True, layout=widgets.Layout(width="70px"))
                      for v in np.ravel(start)]
        self.picture = _picture_widget()
        for box in self.boxes:
            box.observe(lambda change: self.draw(), names="value")
        grid = widgets.GridBox(self.boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 75px)"))
        self.draw()  # before showing it, so that the picture is there from the start
        display(widgets.VBox([grid, self.picture]))

    def kernel(self):
        return np.array([box.value for box in self.boxes]).reshape(3, 3)

    def draw(self):
        kernel = self.kernel()
        filtered = [apply_filter(p, kernel) for p in self.pictures]
        averages = np.array([f.mean() for f in filtered])
        correct, threshold, high_kind = _best_threshold(averages, self.kinds)
        guesses = (averages > threshold) == (high_kind == 1)
        names = [LABELS[kind] for kind in LINE_KINDS[self.level]]

        fig = plt.figure(figsize=(16, 6.5))
        grid = fig.add_gridspec(2, 9, width_ratios=[1] * 6 + [0.3, 2.2, 2.2])
        for i in range(6):  # three pictures of each kind, before and after the filter
            n = [0, 2, 4, 1, 3, 5][i]
            right = guesses[n] == (self.kinds[n] == 1)
            top, bottom = fig.add_subplot(grid[0, i]), fig.add_subplot(grid[1, i])
            top.imshow(self.pictures[n], cmap="gray", vmin=0, vmax=255)
            top.set_title(names[self.kinds[n]], fontsize=10)
            bottom.imshow(filtered[n], cmap="gray", vmin=0, vmax=255)
            bottom.set_title(f"{averages[n]:.0f} {'✓' if right else '✗'}", color="green" if right else "red", fontsize=12)
            for ax in (top, bottom):
                ax.axis("off")
        spread = fig.add_subplot(grid[:, 7:])
        bins = np.linspace(averages.min(), averages.max() + 1e-6, 30)
        for k, colour in ((0, "tab:blue"), (1, "tab:orange")):
            spread.hist(averages[self.kinds == k], bins=bins, alpha=0.7, color=colour, label=names[k])
        spread.axvline(threshold, color="black", linestyle="--", label=LABELS["threshold"])
        spread.set_xlabel(LABELS["average"])
        spread.set_yticks([])
        spread.legend()
        title = f"{LABELS['correct']}: {correct} / {len(self.kinds)}"
        if correct == len(self.kinds):
            title += f"   {LABELS['bravo']}"
        fig.suptitle(title, fontsize=18, color="green" if correct == len(self.kinds) else "black")
        self.picture.value = _png(fig)


def classifier_challenge(level=1, start=NOTHING):
    return ClassifierChallenge(level, start)


FILTERS = {
    "nothing": NOTHING,
    "blur": [[1 / 9] * 3] * 3,
    "sharpen": [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
    "vertical edges": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "horizontal edges": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "all edges": [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]],
}


def learned_filters():
    """The 64 filters in the first layer of ResNet-18, a network that learned from 1.2 million photos (ImageNet).
    Each filter has 7×7 numbers for each of the three colours."""
    import torchvision

    model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
    return model.conv1.weight.detach().numpy()


def show_learned(weights):
    fig, axes = plt.subplots(4, 16, figsize=(16, 4.4))
    for ax, f in zip(axes.flat, weights):
        f = f.transpose(1, 2, 0)
        ax.imshow((f - f.min()) / (f.max() - f.min()))
        ax.axis("off")
    plt.show()


# The network was trained on pictures shifted and scaled by these numbers, so we do the same before using its filters.
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])


def _visible(response):
    """Edges in both directions, scaled so that the strongest ones are white."""
    strength = np.abs(response)
    return np.clip(strength / np.percentile(strength, 99) * 255, 0, 255)


def handmade_vs_learned(rgb, weights):
    """Our edge filters next to the learned filters most like them, each applied to the picture."""
    x = (rgb / 255 - MEAN) / STD
    grey = to_grey(rgb).astype(float)
    ramp = np.tile(np.arange(-3, 4), (7, 1))
    flat = weights.mean(axis=1)  # each filter with its three colours averaged
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), gridspec_kw={"width_ratios": [1, 2, 1, 2]})
    for row, (name, template) in enumerate((("vertical edges", ramp), ("horizontal edges", ramp.T))):
        likeness = np.abs((flat * template).sum(axis=(1, 2))) / np.linalg.norm(flat, axis=(1, 2))
        f = weights[np.argmax(likeness)]
        k = np.asarray(FILTERS[name], dtype=float)
        _cells(axes[row, 0], k, shades=(k - k.min()) / (k.max() - k.min()) * 255, fontsize=11)  # shaded like the network's filter
        axes[row, 0].set_title(f"{LABELS['handmade']}\n{LABELS[name]}")
        axes[row, 1].imshow(_visible(correlate(grey, np.asarray(FILTERS[name], dtype=float), mode="nearest")), cmap="gray", vmin=0, vmax=255)
        axes[row, 1].set_title(LABELS["handmade result"])
        shown = f.transpose(1, 2, 0)
        axes[row, 2].imshow((shown - shown.min()) / (shown.max() - shown.min()))
        axes[row, 2].set_title(LABELS["learned"])
        axes[row, 3].imshow(_visible(sum(correlate(x[..., c], f[c], mode="nearest") for c in range(3))), cmap="gray", vmin=0, vmax=255)
        axes[row, 3].set_title(LABELS["learned result"])
        for ax in (axes[row, 1], axes[row, 2], axes[row, 3]):
            ax.axis("off")
    plt.tight_layout()
    plt.show()

## Toda kako računalnik sploh "vidi in razume" sliko?
Ljudem je vid samoumeven. Ko vidimo mačko na sliki, jo takoj prepoznamo. Toda, kako je slika predstavljena računalniku? Morda malce presenteljivo, je slika predstavljena računalniku kot kup enih številk (katerih vrednost je celo število med 0-255). Radoveden bralec se morda sprašuje: "Kaj pomenijo te številke in kako jih pridobimo iz podane slike?".

> Za lažjo razlago koncepta bomo delali na konkretnem primeru. Poženite spodnjo celico za prikaz slike mačke. Sicer na mesto "cat", vstavite drug niz, npr. "rocket" za prikaz slike rakete.

In [ ]:
picture = choose_picture("cat")  # "cat", "astronaut", "rocket" or "coffee"
show(picture)

In [ ]:
#Naložite lahko tudi svojo sliko namesto predlaganih slik.
my_photo = upload_picture()
if my_photo is not None:
    picture = my_photo
    show(picture)

> Kako računalnik vidi sliko:
- Vsak piksel je celo število med 0 in 255, ki predstavlja svetlost. Piksel z vrednostjo 0 je črn, z vrednostjo 255 je bel, vmes so odtenki sive (teh je več kot 50). Slika je tako predstavljena kot 2D mreža številk. Več pikslov kot imamo za predstavite slike, boljšo resolucijo dobimo. 
- Poženite spodnjo celico. Naša izbrana slika se bo spremenila v mrežo številk. Z drsnikom lahko povečate oz. zmanjšate resolucijo. 
- Vsaka izbrana številka pripada določenemu pikslu na sliki.
- Ali lahko opazite vpliv velikosti številke na svetlost piksla (kvadratka)?

In [ ]:
as_numbers(picture)

Opazimo, da višja številka v kvadratku pripada svetlejšemu pikslu na sliki.

## Kako predstavimo barve računalniku?
- Z eno samo številko na piksel dobimo odtenke sive. Kako pa bi dodali sliki cel barvni razpon?
- Izkaže se, da lahko praktično katerokoli barvo predstavimo kot kombinacijo rdeče, zelene in modre barve. 
- V barvni sliki ima tako vsak piksel predpisane 3 številke, ki povedo koliko rdeče, zelene oz. modre barve dodamo v barvno mešanico za podan piksel
- Z drugimi besedami, piksel je 3 dimenzionalni vektor (R, G, B), kjer R,G,B zasedajo celoštevilske vrednosti med 0 in 255.
- Posledično lahko predstavimo 256 x 256 x 256 (približno 16.7 miljonov) odtenkov barv.
- V spodnji celici se lahko poigraš z mešanjem barv in opazuješ, kakšno mešanico potrebujemo, da pridobimo željeno barvo.

In [ ]:
colour_mixer()

Barvna slika je torej sestavljena iz treh sivih slik, ki so položene ena na drugo. Ena prikazuje, koliko rdeče vsebuje vsak piksel, druga koliko zelene, tretja pa koliko modre.

In [ ]:
channels(picture)

>  Z miško se premikaj skozi sliko in rdeči kvadrat bo sledil. Na desni strani je v kvadratu prikazanih 10 × 10 pikslov, vsak s tremi številkami (R, G, B). 
- Poišči temne in svetle dele slike. Ali opzaiš, kakšne razlike v številkah?

In [ ]:
colour_magnifier(picture)

Spomnimo se slik z začetka. Kako avto iz mreže številk ugotovi, da je pred njim še en avto? Računalnik to naredi po korakih:

<table><tr>
<td align="center" width="50%"><img src="https://img.youtube.com/vi/tlThdr3O5Qo/maxresdefault.jpg" width="100%"><br><b>»avto«</b></td>
</tr></table>

1. številke služijo kot začetna točka za računalnik
2. iz njih razberemo majhne vzorce (robovi, črte) 
3. ti vzorci sestavljajo večje dele (kolo, prtljažnik)
4. iz večjih delov prepoznamo cel predmet (»avto«)

Začetni korak bomo sedaj naredili sami: majhne vzorce na sliki bomo iskali s *filtri*.

## Kaj je filter?

Kako najdemo robove? Spomni se mreže številk s prejšnjega koraka: tam, kjer je rob, se številka nenadoma spremeni, na primer iz 40 v 200. Na enakomernih delih slike pa so sosednje številke skoraj enake. Rob torej najdemo tako, da vsak piksel primerjamo z njegovimi sosedi.

Prav to naredi **filter**: majhna tabela številk, na primer velikosti 3 × 3. Z njim se sprehodimo čez vse piksle slike in pri vsakem:
1. vzamemo piksel in njegovih 8 sosedov, torej prav tako 3 × 3 številke;
2. vsako od njih pomnožimo s številom na istem mestu v filtru;
3. vseh devet zmnožkov seštejemo. Vsota je nova vrednost piksla.

Kaj filter naredi, je odvisno samo od njegovih devetih številk. Z enimi poudari robove, z drugimi pa sliko **zamegli** ali **izostri**, tako kot telefon zamegli ozadje portreta ali kot aplikacija izostri fotografijo. Za vse to računalnik ne potrebuje nič drugega kot preproste računske operacije na številkah slike.

Spodnji urejevalnik se začne s filtrom, ki ne spremeni ničesar: 1 na sredini in 0 okoli.

## Poskusi sam

Poženi spodnjo celico in spreminjaj številke v filtru.
- izziv: Nastavi filter tako, da bodo svetli samo robovi predmetov, vse drugo pa temno.



In [ ]:
editor = filter_editor(picture)

## Izziv 1: skrivno sporočilo

Spodnja slika na videz ni nič posebnega: siva barva, ki se z leve proti desni počasi svetli. A v njej je skrito sporočilo, ki ga na začetku ne vidiš. Poišči devet številk za filter, ki sporočilo razkrijejo, in ga preberi.

<details><summary><b>Namig 1</b></summary>Najprej se znebimo ozadja. Predstavljaj si del ozadja, kjer imajo vsi piksli enako številko, na primer 100. Filter vsako od devetih številk pomnoži s številom v filtru in vse sešteje, zato dobimo 100 × (vsota števil v filtru). Kolikšna mora biti vsota števil v filtru, da dobimo 0, torej črno?</details>

<details><summary><b>Namig 2</b></summary>Skrite črke so narejene iz drobnega vzorca, kot šahovnica: piksli so izmenično malo svetlejši in malo temnejši. Vsak piksel v črkah se zato razlikuje od vseh svojih sosedov. Spomni se filtra, ki je v izzivu »Poskusi sam« pokazal robove: ta primerja vsak piksel z vsemi sosedi okoli njega.</details>

<details><summary><b>Namig 3</b></summary>Večje številke naredijo črke svetlejše.</details>

In [ ]:
secret = secret_picture()
secret_editor = filter_editor(secret)

## Izziv 2: zgradi svoj model

Zdaj zgradimo čisto pravi, čeprav zelo majhen model, ki loči dve vrsti slik. Imamo 100 majhnih slik: na polovici so vodoravne črte, na polovici navpične. Model deluje v treh korakih:
1. na sliko uporabi **tvoj filter**;
2. izračuna **povprečno svetlost** filtrirane slike;
3. povprečje primerja s **pragom**: nad pragom je ena vrsta črt, pod njim druga. Najboljši prag računalnik izbere sam. To je vse, kar se model »nauči«.

S filtrom, ki ne naredi ničesar, model le ugiba in ima prav pri približno polovici slik. **Naloga:** poišči filter, s katerim model pravilno razvrsti vseh 100 slik.

Graf na desni prikazuje povprečja za obe vrsti slik. Ko se modri in oranžni stolpci nehajo prekrivati, ima model vedno prav.

<details><summary><b>Namig 1</b></summary>Filter naj eno vrsto črt »vidi«, za drugo pa naj bo slep.</details>

<details><summary><b>Namig 2</b></summary>Vzdolž vodoravne črte imajo sosednji piksli v isti vrstici enako vrednost. Kaj dobiš, če od piksla odšteješ njegovega levega soseda?</details>

In [ ]:
model = classifier_challenge()

## Kako je s filtri danes? Nevronske mreže same poiščejo najbolj primerne filtre.
- Danes v praksi nihče več ročno ne izbira številk za filtre
- Nevronska mreža ima vpogled v 1.2 miljona slik in filtre (števlike v mreži) prilagodi, tako da prepozna, kaj je v slikah.
- Ko poženeš spodnjo celico, se ti bo prikazalo 64 filtrov veilkosti 7 x 7, ki jih uporablja znana mreža ResNet-18.
- Mreži nihče ni eksplicitno povedal, kakšne številke naj uporabi za filtre. Temveč se je sama naučila, da so spodnji filtri najbolj primerni.

In [ ]:
weights = learned_filters()
show_learned(weights)

## Ali je mreža odkrila iste filtre kot ljudje?

Med 64 filtri mreže so nekateri presenetljivo podobni filtrom, ki so jih ljudje izbrali na roke. Spodnja celica za vsakega od naših dveh filtrov za robove (navpične in vodoravne) poišče filter mreže, ki mu je najbolj podoben, in oba uporabi na tvoji sliki. Poženi spodnjo celico.

**Kako brati sliko** (od leve proti desni):
1. **naš filter**: številke, ki so jih izbrali ljudje. Temen kvadratek pomeni negativno število, svetel pa pozitivno.
2. **po našem filtru**: kaj filter naredi s sliko. Robovi so svetli, enakomerni deli slike pa temni.
3. **najbolj podoben filter mreže**: filter velikosti 7 × 7, ki se ga je mreža naučila sama. Tudi tu temno pomeni negativna, svetlo pa pozitivna števila. Barven je zato, ker ima filter mreže ločene številke za rdečo, zeleno in modro.
4. **po filtru mreže**: kaj filter mreže naredi s tvojo sliko.

**Na kaj bodi pozoren:**
- Pri obeh filtrih je ena stran temna, druga pa svetla. Zato se filter odzove tam, kjer se svetlost hitro spremeni: na robovih.
- Primerjaj drugo in četrto sliko v vrsti. Ali poudarita iste robove?
- Mreži tega ni nihče povedal. Iz 1,2 milijona slik je sama ugotovila, da je iskanje robov dober prvi korak pri prepoznavanju, kaj je na sliki.

In [ ]:
handmade_vs_learned(picture, weights)

## Kaj ni bilo jasno?

Tukaj zapiši, kaj je bilo treba poiskati drugje in česa ni šlo rešiti. Nato to sporoči osebi, ki ti je dala ta zvezek. Tvoja opomba bo pomagala izboljšati naslednjo različico.